# Answers 04 Grid variations

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How does the net change with the supports, the force densities and the loads?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

Fixed nodes, the four corners:
$$ \color{#2a6fdb}{n_0 \, (0, 0, 0) \text{ m}} $$
$$ \color{#2a6fdb}{n_6 \, (6, 0, 0) \text{ m}} $$
$$ \color{#2a6fdb}{n_{42} \, (0, 6, 0) \text{ m}} $$
$$ \color{#2a6fdb}{n_{48} \, (6, 6, 0) \text{ m}} $$

All other 45 nodes are ${\color{#d62828} \text{free}}$.

Load on every free node:
$$ \color{#2e9e4a}{p \, (0, 0, 0) \text{ kN}} $$

The force density:
$$ q = [1, 1, \ldots, 1] \text{ kN/m} $$

Change one thing at a time:

- the boundary conditions in the z axis
- the force densities $q$
- the external loads ${\color{#2e9e4a} p}$

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">


# Network

In [1]:
import numpy as np

# nodes: 6 x 6 cells = 7 x 7 nodes n0..n48, numbered row by row from the bottom left
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

# bars: each node to its right and upper neighbour
edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]

# connectivity matrix
C = np.zeros((len(edges), n*n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

# fixed = 4 corners, free = everything else
fixed = []
free  = []
for k in range(n * n):
    if x[k] in (0, n-1) and y[k] in (0, n-1):
        fixed.append(k)
    else:
        free.append(k)

C_N, C_F = C[:, free], C[:, fixed]

# Solve and Show

The same two steps are needed after every change, so they go into two functions.

$$
\begin{aligned}
x_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_x - C_N^{T}\,Q\,C_F\,x_F\right) \\
y_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_y - C_N^{T}\,Q\,C_F\,y_F\right) \\
z_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_z - C_N^{T}\,Q\,C_F\,z_F\right)
\end{aligned}
$$

In the viewer the free nodes are red and the fixed nodes are blue. A bar is darker when its force $f = q \, l$ is bigger.

In [2]:
from session_py import Color, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, loads green
RED   = Color(0.84, 0.16, 0.16, 1.0)
BLUE  = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY  = Color(0.70, 0.70, 0.70, 1.0)


def form_find(z, q, p_z):
    """Move the free nodes to equilibrium. Returns the new x, y, z and the bar forces f."""
    Q = np.diag(q)
    x_new, y_new, z_new = x.copy(), y.copy(), z.copy()

    # equation 6.32
    A = C_N.T @ Q @ C_N
    x_new[free] = np.linalg.solve(A, 0   - C_N.T @ Q @ C_F @ x[fixed])
    y_new[free] = np.linalg.solve(A, 0   - C_N.T @ Q @ C_F @ y[fixed])
    z_new[free] = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z[fixed])

    # bar forces f = q l
    f = q * np.sqrt((C @ x_new)**2 + (C @ y_new)**2 + (C @ z_new)**2)
    return x_new, y_new, z_new, f


def draw(x, y, z, f, name):
    """Show the net in the viewer. A bar is darker when its force is bigger."""
    colors = []
    for force in f:
        shade = 0.65 - 0.65 * force / f.max()
        colors.append(Color(shade, shade, shade, 1.0))

    session = Session(name)
    session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()), colors, [3.0] * len(edges))

    # node colours
    for k in range(n * n):
        point = session.objects.points[k]
        point.pointcolor = BLUE if k in fixed else RED
        point.width = 14.0 if k in fixed else 7.0

    return show(session, name)

# Start

Flat corners, $q = 1$, no load. The net stays flat. All bars pull equally hard per length, so the edges are sucked inwards.

In [3]:
# force densities and load
q   = np.ones(len(edges))
p_z = np.zeros(len(free))

x1, y1, z1, f1 = form_find(z, q, p_z)
draw(x1, y1, z1, f1, "fdm_exercise_4_start")

# Change the Boundary Conditions

Lift two opposite corners by 4 m. The flat net becomes a saddle.

$$ \color{#2a6fdb}{n_0 \, (0, 0, 4) \text{ m}} $$
$$ \color{#2a6fdb}{n_{48} \, (6, 6, 4) \text{ m}} $$

In [4]:
# lift two opposite corners
z[0]       = 4      # corner (0, 0)
z[n*n - 1] = 4      # corner (6, 6)

x2, y2, z2, f2 = form_find(z, q, p_z)
draw(x2, y2, z2, f2, "fdm_exercise_4_corners")

# Change the Force Densities

The bars on the four edges get $q = 10$ kN/m. The edge cables pull harder, get straighter and stretch the net open.

In [5]:
for e, (a, b) in enumerate(edges):
    if (x[a] == x[b] and x[a] in (0, n-1)) or (y[a] == y[b] and y[a] in (0, n-1)):
        q[e] = 10       # edge cables pull harder

x3, y3, z3, f3 = form_find(z, q, p_z)
draw(x3, y3, z3, f3, "fdm_exercise_4_q")

# Change the Loads

Hang a load on every free node. The net sags. $x$ and $y$ do not change, because the load is vertical.

$$ \color{#2e9e4a}{p \, (0, 0, -0.3) \text{ kN}} $$

In [6]:
p_z = -0.3 * np.ones(len(free))

x4, y4, z4, f4 = form_find(z, q, p_z)
draw(x4, y4, z4, f4, "fdm_exercise_4_load")

In [7]:
middle = (n * n) // 2     # node n24 in the middle of the net
print("height of the middle node before the load:", round(z3[middle], 2), "m")
print("height of the middle node with the load:  ", round(z4[middle], 2), "m")
print("biggest bar force before the load:", round(f3.max(), 2), "kN")
print("biggest bar force with the load:  ", round(f4.max(), 2), "kN")

height of the middle node before the load: 2.0 m
height of the middle node with the load:   0.96 m
biggest bar force before the load: 12.48 kN
biggest bar force with the load:   13.5 kN


# Observation

Every viewer shows a net in exact equilibrium, found with one linear solve. Now change the numbers yourself: other corners, other $q$, a load in the other direction ($p_z = +0.3$ gives a net shaped like a dome).